# Model B — Multi-Label Citrus Disease Classifier

**Architecture:** ConvNeXt-Small (pretrained ImageNet)  
**Task:** Detect co-occurring diseases from a single leaf image  
**Loss:** Asymmetric Loss (ASL) — handles multi-label class imbalance  
**Dataset:** All 10,913 images from `multilabel.csv` (single + multi-label)  
**Target:** mAP > 0.90, macro F1 > 0.85

---

### Notebook structure
1. Setup and imports  
2. Configuration  
3. Dataset class  
4. Augmentation pipeline  
5. Asymmetric Loss (ASL)  
6. MixUp / CutMix  
7. Model architecture  
8. Metrics  
9. Training loop  
10. Per-class threshold tuning  
11. Final evaluation  
12. Plots  
13. Save model for deployment

## Cell 1 — Install dependencies

In [1]:
import subprocess, sys

pkgs = ["torch", "torchvision", "scikit-learn", "pandas", "tqdm", "matplotlib", "seaborn", "Pillow"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg])

# Fix SSL for Mac (model download)
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

print("✓ All dependencies ready")


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


✓ All dependencies ready



[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


## Cell 2 — Imports

In [2]:
import os, json, time, csv, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from PIL import Image
from contextlib import nullcontext
from sklearn.metrics import (
    f1_score, average_precision_score,
    hamming_loss, classification_report,
    confusion_matrix
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm

warnings.filterwarnings("ignore")
print(f"PyTorch version : {torch.__version__}")
print(f"Torchvision     : {__import__('torchvision').__version__}")

PyTorch version : 2.12.0
Torchvision     : 0.27.0


## Cell 3 — Configuration

**Change `DATASET_DIR` and `CSV_PATH` to match your folder structure.**

In [3]:
# ── PATHS ─────────────────────────────────────────────────────────────────────
DATASET_DIR    = "dataset_balanced"               # folder with train/val/test subfolders
CSV_PATH       = "dataset_balanced/multilabel.csv"
OUTPUT_DIR     = "model_b_multilabel"          # all outputs saved here
CHECKPOINT     = f"{OUTPUT_DIR}/best_model.pth"
THRESH_FILE    = f"{OUTPUT_DIR}/thresholds.json"
LOG_FILE       = f"{OUTPUT_DIR}/training_log.json"
PLOTS_DIR      = f"{OUTPUT_DIR}/plots"

# ── MODEL ─────────────────────────────────────────────────────────────────────
IMG_SIZE       = 224    # ConvNeXt-S native size
BATCH_SIZE     = 32
NUM_WORKERS    = 0      # 0 for Mac MPS; set 4 for CUDA

# ── TRAINING ──────────────────────────────────────────────────────────────────
EPOCHS         = 50
WARMUP_EPOCHS  = 5
LR_HEAD        = 3e-4
LR_BACKBONE    = 5e-5
WEIGHT_DECAY   = 1e-4
PATIENCE       = 10     # early stopping on val mAP
DEFAULT_THRESH = 0.5

# ── DATASET STATS (from step4_compute_mean_std.py) ────────────────────────────
MEAN = [0.6753, 0.7039, 0.5966]
STD  = [0.3517, 0.3215, 0.4195]

# ── CLASSES ───────────────────────────────────────────────────────────────────
ALL_CLASSES = [
    "Anthracnose", "BacterialBlight", "BlackSpot", "Canker",
    "CurlVirus", "DeficiencyLeaf", "DryLeaf", "Greening",
    "Healthy", "LeafMiner", "Melanose", "SootyMould", "SpiderMites"
]
NUM_CLASSES = len(ALL_CLASSES)

# Create output dirs
for d in [OUTPUT_DIR, PLOTS_DIR]:
    Path(d).mkdir(parents=True, exist_ok=True)

print(f"Classes   : {NUM_CLASSES}")
print(f"Dataset   : {DATASET_DIR}")
print(f"CSV       : {CSV_PATH}")
print(f"Output    : {OUTPUT_DIR}")

Classes   : 13
Dataset   : dataset_balanced
CSV       : dataset_balanced/multilabel.csv
Output    : model_b_multilabel


## Cell 4 — Device setup

In [4]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def autocast_ctx(device):
    """torch.autocast works on cuda/cpu. MPS uses nullcontext (float32)."""
    if device.type in ("cuda", "cpu"):
        return torch.autocast(device_type=device.type)
    return nullcontext()

DEVICE = get_device()
PIN_MEMORY  = DEVICE.type == "cuda"

print(f"Device     : {DEVICE}")
print(f"Pin memory : {PIN_MEMORY}")
print(f"Workers    : {NUM_WORKERS}")

Device     : mps
Pin memory : False
Workers    : 0


## Cell 5 — Dataset class

Reads from `multilabel.csv`. Every row is one image with a 13-column binary label vector.  
Works for both single-label images (one 1 in the vector) and multi-label images (multiple 1s).

In [5]:
class CitrusMultiLabelDataset(Dataset):
    """
    Loads all images from multilabel.csv for a given split.

    Returns:
        image  : (3, IMG_SIZE, IMG_SIZE) float tensor
        labels : (13,) float tensor — multi-hot binary vector
                 e.g. Anthracnose + Melanose → [1,0,0,0,0,0,0,0,0,0,1,0,0]
    """

    def __init__(self, split: str, transform=None):
        self.transform = transform
        self.samples   = []

        if not Path(CSV_PATH).exists():
            raise FileNotFoundError(f"multilabel.csv not found: {CSV_PATH}")

        df       = pd.read_csv(CSV_PATH)
        split_df = df[df["split"] == split].reset_index(drop=True)

        missing = 0
        for _, row in split_df.iterrows():
            img_path = Path(DATASET_DIR) / row["image_path"]
            if not img_path.exists():
                missing += 1
                continue
            label_vec = torch.tensor(
                [float(row[cls]) for cls in ALL_CLASSES],
                dtype=torch.float32
            )
            self.samples.append((img_path, label_vec))

        multi  = sum(1 for _, lv in self.samples if lv.sum() > 1)
        single = len(self.samples) - multi
        print(f"  [{split:5s}] {len(self.samples):6d} images  "
              f"| single-label: {single:5d}  multi-label: {multi:4d}  "
              f"| missing: {missing}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label_vec = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label_vec


# ── Quick check ───────────────────────────────────────────────────────────────
print("Dataset loading check:")
_ = CitrusMultiLabelDataset("train")
_ = CitrusMultiLabelDataset("val")
_ = CitrusMultiLabelDataset("test")

Dataset loading check:
  [train]   8340 images  | single-label:  8041  multi-label:  299  | missing: 0
  [val  ]   1275 images  | single-label:  1219  multi-label:   56  | missing: 0
  [test ]   1298 images  | single-label:  1235  multi-label:   63  | missing: 0


## Cell 6 — Augmentation pipeline

Train: aggressive colour + spatial augmentation optimised for plant disease images.  
Val/Test: deterministic resize + normalise only.

In [6]:
def get_transforms(split: str) -> transforms.Compose:
    normalize = transforms.Normalize(mean=MEAN, std=STD)

    if split == "train":
        return transforms.Compose([
            transforms.Resize((IMG_SIZE + 32, IMG_SIZE + 32)),
            transforms.RandomCrop(IMG_SIZE),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(degrees=30),
            transforms.ColorJitter(
                brightness=0.35,
                contrast=0.35,
                saturation=0.30,
                hue=0.12,
            ),
            # RandAugment learns the best augmentation policy automatically
            transforms.RandAugment(num_ops=2, magnitude=9),
            transforms.ToTensor(),
            normalize,
            # Random erasing simulates partial occlusion of lesion areas
            transforms.RandomErasing(p=0.25, scale=(0.02, 0.15)),
        ])
    else:
        return transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor(),
            normalize,
        ])


print("Train transforms:")
for t in get_transforms("train").transforms:
    print(f"  {t.__class__.__name__}")
print("\nVal/Test transforms:")
for t in get_transforms("val").transforms:
    print(f"  {t.__class__.__name__}")

Train transforms:
  Resize
  RandomCrop
  RandomHorizontalFlip
  RandomVerticalFlip
  RandomRotation
  ColorJitter
  RandAugment
  ToTensor
  Normalize
  RandomErasing

Val/Test transforms:
  Resize
  ToTensor
  Normalize


## Cell 7 — MixUp and CutMix

Applied at batch level during training. Works correctly with multi-hot label vectors —  
the mixed label is a weighted average of two binary vectors, which BCEWithLogitsLoss handles natively.

In [7]:
def mixup(imgs: torch.Tensor, labels: torch.Tensor,
          alpha: float = 0.4) -> tuple:
    """
    MixUp: blend two images and their label vectors linearly.
    Works for both single-label (one-hot) and multi-label (multi-hot) vectors.

    Returns: (mixed_imgs, mixed_labels)
    """
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(imgs.size(0), device=imgs.device)
    mixed_imgs   = lam * imgs + (1 - lam) * imgs[idx]
    mixed_labels = lam * labels + (1 - lam) * labels[idx]
    return mixed_imgs, mixed_labels


def cutmix(imgs: torch.Tensor, labels: torch.Tensor,
           alpha: float = 1.0) -> tuple:
    """
    CutMix: paste a random rectangular patch from one image into another.
    Labels are mixed proportionally to the area of the patch.

    Returns: (mixed_imgs, mixed_labels)
    """
    lam   = np.random.beta(alpha, alpha)
    idx   = torch.randperm(imgs.size(0), device=imgs.device)
    _, _, H, W = imgs.shape
    cut_h = int(H * np.sqrt(1 - lam))
    cut_w = int(W * np.sqrt(1 - lam))
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    mixed          = imgs.clone()
    mixed[:, :, y1:y2, x1:x2] = imgs[idx, :, y1:y2, x1:x2]
    lam_actual     = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    mixed_labels   = lam_actual * labels + (1 - lam_actual) * labels[idx]
    return mixed, mixed_labels


def apply_batch_augmentation(imgs: torch.Tensor,
                              labels: torch.Tensor,
                              p: float = 0.6) -> tuple:
    """
    Apply MixUp or CutMix with probability p.
    50% chance of each when triggered.
    """
    if np.random.rand() < p:
        if np.random.rand() < 0.5:
            return mixup(imgs, labels, alpha=0.4)
        else:
            return cutmix(imgs, labels, alpha=1.0)
    return imgs, labels


print("✓ MixUp and CutMix defined")
print("  Applied to 60% of training batches")
print("  MixUp alpha=0.4 | CutMix alpha=1.0")

✓ MixUp and CutMix defined
  Applied to 60% of training batches
  MixUp alpha=0.4 | CutMix alpha=1.0


## Cell 8 — Asymmetric Loss (ASL)

Standard BCE treats positive and negative examples equally.  
In multi-label with 13 classes, ~96% of labels are 0 (negative) — BCE wastes gradient on easy negatives.  

ASL fixes this by:  
- Down-weighting easy negatives via `gamma_neg=4`  
- Keeping full focus on positives via `gamma_pos=1`  
- Shifting negative probabilities down by `clip=0.05`

Reference: Ridnik et al., ASL paper, 2021

In [8]:
class AsymmetricLoss(nn.Module):
    """
    Asymmetric Loss for multi-label classification.

    Args:
        gamma_neg : focusing exponent for negatives (default 4)
        gamma_pos : focusing exponent for positives (default 1)
        clip      : probability shift for negatives (default 0.05)

    Intuition:
        - Positives get standard focal treatment (weight = (1-p)^gamma_pos)
        - Negatives get shifted DOWN by 'clip' first, then focused harder
        - This prevents the loss from being dominated by easy negatives
    """

    def __init__(self, gamma_neg: float = 4,
                 gamma_pos: float = 1,
                 clip: float = 0.05):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip      = clip

    def forward(self, logits: torch.Tensor,
                targets: torch.Tensor) -> torch.Tensor:
        probs     = torch.sigmoid(logits)
        probs_neg = (probs - self.clip).clamp(min=0)   # shift negatives down

        # Log probabilities
        loss_pos = targets       * torch.log(probs.clamp(min=1e-8))
        loss_neg = (1 - targets) * torch.log((1 - probs_neg).clamp(min=1e-8))
        loss     = loss_pos + loss_neg

        # Asymmetric focusing weights
        pt_pos = probs       * targets
        pt_neg = probs_neg   * (1 - targets)
        weight = (targets       * (1 - pt_pos) ** self.gamma_pos +
                  (1 - targets) * pt_neg        ** self.gamma_neg)

        loss = -loss * weight
        return loss.mean()


# Test ASL
asl_test = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)
dummy_logits  = torch.randn(4, 13)
dummy_targets = torch.zeros(4, 13)
dummy_targets[0, 0] = 1.0
dummy_targets[0, 10] = 1.0  # multi-label: Anthracnose + Melanose
loss_val = asl_test(dummy_logits, dummy_targets)
print(f"✓ ASL defined — test loss: {loss_val.item():.4f}")
print(f"  gamma_neg=4 (down-weight easy negatives)")
print(f"  gamma_pos=1 (standard focal on positives)")
print(f"  clip=0.05   (shift negatives away from decision boundary)")

✓ ASL defined — test loss: 0.1226
  gamma_neg=4 (down-weight easy negatives)
  gamma_pos=1 (standard focal on positives)
  clip=0.05   (shift negatives away from decision boundary)


## Cell 9 — Class weights for pos_weight

Computes per-class positive weights from the training split.  
`pos_weight[c] = neg_count[c] / pos_count[c]`  
Used as a fallback BCE criterion alongside ASL.

In [9]:
def compute_pos_weight(split: str = "train") -> torch.Tensor:
    df  = pd.read_csv(CSV_PATH)
    df  = df[df["split"] == split]
    pos = df[ALL_CLASSES].sum(axis=0).values.astype(float)
    neg = len(df) - pos
    pw  = neg / np.maximum(pos, 1)
    print(f"  pos_weight range: [{pw.min():.2f}, {pw.max():.2f}]")
    return torch.tensor(pw, dtype=torch.float32)


print("Computing class weights from training split...")
POS_WEIGHT = compute_pos_weight("train")
print(f"  Shape: {POS_WEIGHT.shape}")

# Show per-class weights
df_tmp = pd.read_csv(CSV_PATH)
df_tmp = df_tmp[df_tmp["split"] == "train"]
print(f"\n  {'Class':<22} {'Positives':>10}  {'Weight':>8}")
print("  " + "─" * 42)
for i, cls in enumerate(ALL_CLASSES):
    pos_count = int(df_tmp[cls].sum())
    print(f"  {cls:<22} {pos_count:>10}  {POS_WEIGHT[i]:>8.2f}")

Computing class weights from training split...
  pos_weight range: [4.08, 15.68]
  Shape: torch.Size([13])

  Class                   Positives    Weight
  ──────────────────────────────────────────
  Anthracnose                  1643      4.08
  BacterialBlight               500     15.68
  BlackSpot                     788      9.58
  Canker                        620     12.45
  CurlVirus                     513     15.26
  DeficiencyLeaf                510     15.35
  DryLeaf                       500     15.68
  Greening                      575     13.50
  Healthy                       551     14.14
  LeafMiner                     500     15.68
  Melanose                      964      7.65
  SootyMould                    500     15.68
  SpiderMites                   500     15.68


## Cell 10 — Model architecture (ConvNeXt-Small)

ConvNeXt-S pretrained on ImageNet, final classification head replaced with 13 outputs.  
Phase 1 (warmup): backbone frozen, only head trained.  
Phase 2 (finetune): all layers trained with differential learning rates.

In [10]:
def build_model() -> nn.Module:
    """
    ConvNeXt-Small:
      - 50M parameters
      - 83.1% ImageNet top-1
      - Pure CNN — no patches, better texture features than ViT
      - Designed for small-to-medium datasets
    """
    weights = models.ConvNeXt_Small_Weights.IMAGENET1K_V1
    model   = models.convnext_small(weights=weights)

    # Freeze backbone for warmup phase
    for p in model.features.parameters():
        p.requires_grad = False

    # Replace final classification head
    # ConvNeXt-S classifier: Sequential(LayerNorm, Flatten, Linear(768, 1000))
    in_features = model.classifier[2].in_features
    model.classifier[2] = nn.Linear(in_features, NUM_CLASSES)
    # No sigmoid — BCEWithLogitsLoss and ASL apply it internally

    return model


model = build_model().to(DEVICE)

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen    = total - trainable

print(f"Model          : ConvNeXt-Small")
print(f"Total params   : {total:,}")
print(f"Trainable      : {trainable:,}  (head only — warmup phase)")
print(f"Frozen         : {frozen:,}    (backbone)")
print(f"Output size    : {NUM_CLASSES} logits (no sigmoid)")
print(f"Device         : {DEVICE}")

Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /Users/udayapragnagangavaram/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [00:02<00:00, 97.0MB/s] 


Model          : ConvNeXt-Small
Total params   : 49,464,685
Trainable      : 11,533  (head only — warmup phase)
Frozen         : 49,453,152    (backbone)
Output size    : 13 logits (no sigmoid)
Device         : mps


## Cell 11 — DataLoaders

In [11]:
print("Building datasets...")
train_ds = CitrusMultiLabelDataset("train", get_transforms("train"))
val_ds   = CitrusMultiLabelDataset("val",   get_transforms("val"))
test_ds  = CitrusMultiLabelDataset("test",  get_transforms("val"))

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY, drop_last=True
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY
)

print(f"\nTrain batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

# Verify batch shape
imgs, lbls = next(iter(train_loader))
print(f"\nBatch image shape : {tuple(imgs.shape)}")
print(f"Batch label shape : {tuple(lbls.shape)}")
print(f"Pixel range       : [{imgs.min():.3f}, {imgs.max():.3f}]")
print(f"Label range       : [{lbls.min():.1f}, {lbls.max():.1f}]")

Building datasets...
  [train]   8340 images  | single-label:  8041  multi-label:  299  | missing: 0
  [val  ]   1275 images  | single-label:  1219  multi-label:   56  | missing: 0
  [test ]   1298 images  | single-label:  1235  multi-label:   63  | missing: 0

Train batches : 260
Val batches   : 40
Test batches  : 41

Batch image shape : (32, 3, 224, 224)
Batch label shape : (32, 13)
Pixel range       : [-2.189, 0.962]
Label range       : [0.0, 1.0]


## Cell 12 — Metrics

Multi-label requires different metrics than single-label accuracy.  
We track: mAP, macro F1, micro F1, Hamming loss, exact match ratio.

In [12]:
def compute_metrics(labels: np.ndarray,
                    probs: np.ndarray,
                    thresholds: np.ndarray = None) -> dict:
    """
    labels     : (N, 13) binary ground truth
    probs      : (N, 13) sigmoid probabilities
    thresholds : (13,)  per-class thresholds, defaults to 0.5

    Returns dict with all multi-label metrics.
    """
    if thresholds is None:
        thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)

    preds = (probs >= thresholds).astype(int)

    # Mean Average Precision — primary metric
    per_ap = []
    per_ap_dict = {}
    for c, cls in enumerate(ALL_CLASSES):
        if labels[:, c].sum() > 0:
            ap = average_precision_score(labels[:, c], probs[:, c])
            per_ap.append(ap)
            per_ap_dict[cls] = round(float(ap), 4)

    mAP = float(np.mean(per_ap)) if per_ap else 0.0

    return {
        "mAP"          : round(mAP, 5),
        "macro_f1"     : round(f1_score(labels, preds, average="macro",  zero_division=0), 5),
        "micro_f1"     : round(f1_score(labels, preds, average="micro",  zero_division=0), 5),
        "hamming_loss" : round(hamming_loss(labels, preds), 5),
        "exact_match"  : round(float((preds == labels).all(axis=1).mean()), 5),
        "per_class_ap" : per_ap_dict,
    }


print("Metrics computed:")
print("  mAP          — Mean Average Precision (primary metric)")
print("  macro_f1     — unweighted average F1 per class")
print("  micro_f1     — global TP/FP/FN aggregated")
print("  hamming_loss — fraction of wrong labels (lower = better)")
print("  exact_match  — % images where ALL labels correct")

Metrics computed:
  mAP          — Mean Average Precision (primary metric)
  macro_f1     — unweighted average F1 per class
  micro_f1     — global TP/FP/FN aggregated
  hamming_loss — fraction of wrong labels (lower = better)
  exact_match  — % images where ALL labels correct


## Cell 13 — Train and evaluation functions

In [13]:
def train_one_epoch(model, loader, criterion, optimizer,
                    use_mix: bool = True) -> dict:
    model.train()
    total_loss = n_samples = 0

    for imgs, labels in tqdm(loader, desc="  train", leave=False):
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        # Apply MixUp / CutMix
        if use_mix:
            imgs, labels = apply_batch_augmentation(imgs, labels, p=0.6)

        optimizer.zero_grad()
        with autocast_ctx(DEVICE):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        loss.backward()
        # Gradient clipping — prevents exploding gradients
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * imgs.size(0)
        n_samples  += imgs.size(0)

    return {"loss": total_loss / max(n_samples, 1)}


@torch.no_grad()
def evaluate(model, loader,
             thresholds: np.ndarray = None) -> tuple:
    """
    Returns (metrics_dict, all_labels, all_probs)
    """
    model.eval()
    all_labels = []
    all_probs  = []

    for imgs, labels in tqdm(loader, desc="  eval ", leave=False):
        imgs = imgs.to(DEVICE, non_blocking=True)
        with autocast_ctx(DEVICE):
            logits = model(imgs)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(labels.numpy())

    labels_np = np.concatenate(all_labels, axis=0)
    probs_np  = np.concatenate(all_probs,  axis=0)
    metrics   = compute_metrics(labels_np, probs_np, thresholds)
    return metrics, labels_np, probs_np


print("✓ Training and evaluation functions defined")

✓ Training and evaluation functions defined


## Cell 14 — Loss, optimiser, scheduler setup

In [14]:
# Loss: Asymmetric Loss
criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)

# Phase 1 optimiser: head only
head_params  = [p for p in model.classifier.parameters()]
head_optimizer = torch.optim.AdamW(
    head_params, lr=LR_HEAD, weight_decay=WEIGHT_DECAY
)

# Phase 1 scheduler: linear warmup
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    head_optimizer, start_factor=0.1, end_factor=1.0,
    total_iters=WARMUP_EPOCHS
)

print(f"Loss           : AsymmetricLoss (gamma_neg=4, gamma_pos=1)")
print(f"Phase 1 optim  : AdamW on head only (lr={LR_HEAD})")
print(f"Phase 1 sched  : LinearLR warmup over {WARMUP_EPOCHS} epochs")
print(f"Phase 2 optim  : AdamW (backbone lr={LR_BACKBONE}, head lr={LR_HEAD})")
print(f"Phase 2 sched  : CosineAnnealingLR")

Loss           : AsymmetricLoss (gamma_neg=4, gamma_pos=1)
Phase 1 optim  : AdamW on head only (lr=0.0003)
Phase 1 sched  : LinearLR warmup over 5 epochs
Phase 2 optim  : AdamW (backbone lr=5e-05, head lr=0.0003)
Phase 2 sched  : CosineAnnealingLR


## Cell 15 — Phase 1: Warmup (head only, backbone frozen)

In [15]:
print("=" * 70)
print(f"  PHASE 1 — Warmup ({WARMUP_EPOCHS} epochs, backbone frozen)")
print("=" * 70)
print(f"  {'Ep':>3}  {'Train Loss':>11}  {'Val mAP':>9}  {'Macro F1':>10}  {'Time':>6}")

warmup_log = []

for epoch in range(1, WARMUP_EPOCHS + 1):
    t0 = time.time()
    tr = train_one_epoch(model, train_loader, criterion,
                         head_optimizer, use_mix=False)
    vm, _, _ = evaluate(model, val_loader)
    warmup_scheduler.step()

    elapsed = time.time() - t0
    print(f"  {epoch:>3}  {tr['loss']:>11.4f}  {vm['mAP']:>9.4f}  "
          f"{vm['macro_f1']:>10.4f}  [{elapsed:.0f}s]")

    warmup_log.append({
        "epoch"     : epoch,
        "phase"     : "warmup",
        "train_loss": round(tr["loss"], 5),
        "val_mAP"   : round(vm["mAP"], 5),
        "val_macro_f1": round(vm["macro_f1"], 5),
    })

print(f"\n✓ Warmup complete. Backbone unfreezing for phase 2...")

  PHASE 1 — Warmup (5 epochs, backbone frozen)
   Ep   Train Loss    Val mAP    Macro F1    Time


    1       0.0498     0.2258      0.2284  [220s]


    2       0.0353     0.6074      0.4768  [226s]


    3       0.0269     0.7354      0.5356  [226s]


    4       0.0227     0.7994      0.5836  [217s]


    5       0.0202     0.8281      0.6272  [212s]

✓ Warmup complete. Backbone unfreezing for phase 2...


## Cell 16 — Phase 2: Full fine-tuning with early stopping

In [16]:
# Unfreeze all parameters
for p in model.parameters():
    p.requires_grad = True

# Differential LR: backbone gets 6× lower LR than head
optimizer = torch.optim.AdamW([
    {"params": model.features.parameters(),    "lr": LR_BACKBONE},
    {"params": model.classifier.parameters(),  "lr": LR_HEAD},
], weight_decay=WEIGHT_DECAY)

remaining   = EPOCHS - WARMUP_EPOCHS
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=remaining, eta_min=1e-6
)

trainable_p2 = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable params (phase 2): {trainable_p2:,} (all layers)")
print(f"Backbone LR : {LR_BACKBONE}")
print(f"Head LR     : {LR_HEAD}")

print("\n" + "=" * 70)
print(f"  PHASE 2 — Full fine-tune ({remaining} epochs, all layers)")
print(f"  Early stopping patience = {PATIENCE} epochs on val mAP")
print("=" * 70)
print(f"  {'Ep':>3}  {'Train Loss':>11}  {'Val mAP':>9}  "
      f"{'Macro F1':>10}  {'Exact':>7}  {'Time':>6}")

best_mAP         = 0.0
patience_counter = 0
finetune_log     = []

for epoch in range(1, remaining + 1):
    t0     = time.time()
    ep_abs = epoch + WARMUP_EPOCHS

    tr = train_one_epoch(model, train_loader, criterion,
                         optimizer, use_mix=True)
    vm, _, _ = evaluate(model, val_loader)
    scheduler.step()

    elapsed = time.time() - t0
    marker  = ""

    if vm["mAP"] > best_mAP:
        best_mAP = vm["mAP"]
        torch.save({
            "epoch"        : ep_abs,
            "model_state"  : model.state_dict(),
            "val_mAP"      : vm["mAP"],
            "val_macro_f1" : vm["macro_f1"],
            "classes"      : ALL_CLASSES,
            "img_size"     : IMG_SIZE,
            "mean"         : MEAN,
            "std"          : STD,
        }, CHECKPOINT)
        marker           = "  ✓ saved"
        patience_counter = 0
    else:
        patience_counter += 1

    print(f"  {ep_abs:>3}  {tr['loss']:>11.4f}  {vm['mAP']:>9.4f}  "
          f"{vm['macro_f1']:>10.4f}  {vm['exact_match']:>7.4f}  "
          f"[{elapsed:.0f}s]{marker}")

    finetune_log.append({
        "epoch"        : ep_abs,
        "phase"        : "finetune",
        "train_loss"   : round(tr["loss"], 5),
        "val_mAP"      : round(vm["mAP"], 5),
        "val_macro_f1" : round(vm["macro_f1"], 5),
        "val_exact"    : round(vm["exact_match"], 5),
        "hamming_loss" : round(vm["hamming_loss"], 5),
    })

    if patience_counter >= PATIENCE:
        print(f"\n  Early stopping at epoch {ep_abs} (patience={PATIENCE})")
        break

# Save full log
full_log = warmup_log + finetune_log
with open(LOG_FILE, "w") as f:
    json.dump(full_log, f, indent=2)

print(f"\n  Best val mAP : {best_mAP:.4f}")
print(f"  Checkpoint   : {CHECKPOINT}")
print(f"  Training log : {LOG_FILE}")

Trainable params (phase 2): 49,464,685 (all layers)
Backbone LR : 5e-05
Head LR     : 0.0003

  PHASE 2 — Full fine-tune (45 epochs, all layers)
  Early stopping patience = 10 epochs on val mAP
   Ep   Train Loss    Val mAP    Macro F1    Exact    Time


    6       0.0220     0.9220      0.8203   0.6078  [599s]  ✓ saved


    7       0.0191     0.9436      0.8710   0.6894  [571s]  ✓ saved


    8       0.0165     0.9529      0.8778   0.7349  [556s]  ✓ saved


    9       0.0158     0.9607      0.8941   0.7506  [546s]  ✓ saved


   10       0.0144     0.9584      0.9022   0.7757  [545s]


... (30 outputs left out)

   41       0.0124     0.9862      0.9594   0.9255  [558s]


   42       0.0110     0.9862      0.9600   0.9263  [558s]


   43       0.0113     0.9858      0.9573   0.9239  [563s]


   44       0.0096     0.9859      0.9578   0.9247  [556s]

  Early stopping at epoch 44 (patience=10)

  Best val mAP : 0.9865
  Checkpoint   : model_b_multilabel/best_model.pth
  Training log : model_b_multilabel/training_log.json


## Cell 17 — Per-class threshold tuning

Instead of using 0.5 for all classes, we find the threshold per class  
that maximises F1 on the validation set.

In [ ]:
def tune_thresholds(labels: np.ndarray,
                    probs: np.ndarray) -> np.ndarray:
    """
    Per-class threshold tuning:
    Search t in [0.1, 0.90] at 0.05 steps.
    Pick t that maximises F1 per class.
    """
    best_thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)
    candidates      = np.arange(0.10, 0.91, 0.05)

    print(f"  {'Class':<22}  {'Best t':>7}  {'Best F1':>9}  {'Change from 0.5'}>")
    print("  " + "─" * 60)

    for c, cls in enumerate(ALL_CLASSES):
        if labels[:, c].sum() == 0:
            continue
        best_f1 = 0.0
        best_t  = DEFAULT_THRESH
        for t in candidates:
            preds = (probs[:, c] >= t).astype(int)
            f1    = f1_score(labels[:, c], preds, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t  = t
        best_thresholds[c] = best_t
        change = "▲" if best_t > 0.5 else ("▼" if best_t < 0.5 else "=")
        print(f"  {cls:<22}  {best_t:>7.2f}  {best_f1:>9.4f}  {change}")

    return best_thresholds


# Load best checkpoint
print(f"Loading best checkpoint: {CHECKPOINT}")
ckpt = torch.load(CHECKPOINT, map_location=DEVICE)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"  Loaded epoch {ckpt['epoch']} | val_mAP={ckpt['val_mAP']:.4f}")

print("\nTuning thresholds on validation set...")
_, val_labels, val_probs = evaluate(model, val_loader)
THRESHOLDS = tune_thresholds(val_labels, val_probs)

# Save thresholds
thresh_dict = {cls: float(t) for cls, t in zip(ALL_CLASSES, THRESHOLDS)}
with open(THRESH_FILE, "w") as f:
    json.dump(thresh_dict, f, indent=2)
print(f"\n✓ Thresholds saved → {THRESH_FILE}")

## Cell 18 — Final test set evaluation

In [ ]:
print("\nEvaluating on test set...")
_, test_labels, test_probs = evaluate(model, test_loader)

# Metrics with default threshold (0.5)
metrics_def   = compute_metrics(test_labels, test_probs)
# Metrics with tuned per-class thresholds
metrics_tuned = compute_metrics(test_labels, test_probs, THRESHOLDS)

print(f"\n{'=' * 65}")
print(f"  TEST SET RESULTS")
print(f"{'=' * 65}")
print(f"  {'Metric':<25}  {'Default (t=0.5)':>16}  {'Tuned thresh':>14}")
print(f"  {'─' * 60}")

for key in ["mAP", "macro_f1", "micro_f1", "hamming_loss", "exact_match"]:
    d = metrics_def.get(key, 0)
    t = metrics_tuned.get(key, 0)
    better = "↑" if (
        (key != "hamming_loss" and t > d) or
        (key == "hamming_loss" and t < d)
    ) else ""
    print(f"  {key:<25}  {d:>16.4f}  {t:>14.4f}  {better}")

print(f"\n  Per-class Average Precision (tuned):")
print(f"  {'Class':<22}  {'AP':>8}  {'Bar'}")
print(f"  {'─' * 50}")
for cls, ap in sorted(metrics_tuned["per_class_ap"].items(),
                       key=lambda x: -x[1]):
    bar   = "█" * int(ap * 20)
    emoji = "✓" if ap >= 0.85 else "~" if ap >= 0.70 else "✗"
    print(f"  {emoji} {cls:<20}  {ap:>6.4f}  {bar}")

print(f"{'=' * 65}")

## Cell 19 — Classification report (sklearn)

In [ ]:
test_preds = (test_probs >= THRESHOLDS).astype(int)

report = classification_report(
    test_labels, test_preds,
    target_names=ALL_CLASSES,
    digits=4, zero_division=0
)
print(report)

# Save report
report_path = f"{OUTPUT_DIR}/classification_report.txt"
with open(report_path, "w") as f:
    f.write(f"Model B — Multi-Label Classification Report\n")
    f.write(f"Architecture : ConvNeXt-Small\n")
    f.write(f"Best epoch   : {ckpt['epoch']}\n")
    f.write(f"Val mAP      : {ckpt['val_mAP']:.4f}\n\n")
    f.write(f"Tuned Thresholds:\n")
    for cls, t in thresh_dict.items():
        f.write(f"  {cls:<22} {t:.2f}\n")
    f.write(f"\n{report}")
print(f"\n✓ Report saved → {report_path}")

## Cell 20 — Training curves plot

In [ ]:
log      = json.load(open(LOG_FILE))
epochs   = [e["epoch"]       for e in log]
tr_loss  = [e["train_loss"]  for e in log]
va_map   = [e["val_mAP"]     for e in log]
va_f1    = [e["val_macro_f1"]for e in log]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Model B — ConvNeXt-S Multi-Label Training", fontsize=14)

axes[0].plot(epochs, tr_loss, marker="o", markersize=3,
             color="#378ADD", label="Train loss")
axes[0].axvline(x=WARMUP_EPOCHS, color="gray", linestyle="--",
                linewidth=1, label="Warmup end")
axes[0].set_title("ASL Training Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs, va_map, marker="o", markersize=3,
             color="#1D9E75", label="Val mAP")
axes[1].plot(epochs, va_f1, marker="s", markersize=3,
             color="#7F77DD", label="Val Macro F1")
axes[1].axvline(x=WARMUP_EPOCHS, color="gray", linestyle="--",
                linewidth=1, label="Warmup end")
axes[1].set_title("Validation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Training curves → {PLOTS_DIR}/training_curves.png")

## Cell 21 — Per-class AP bar chart

In [ ]:
ap_dict = metrics_tuned["per_class_ap"]
classes = list(ap_dict.keys())
values  = list(ap_dict.values())
colors  = ["#1D9E75" if v >= 0.85 else
           "#EF9F27" if v >= 0.70 else
           "#E24B4A" for v in values]

fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(classes, values, color=colors, edgecolor="white", linewidth=0.5)
ax.axhline(np.mean(values), color="#378ADD", linestyle="--",
           linewidth=1.5, label=f"mAP = {np.mean(values):.3f}")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Average Precision")
ax.set_title("Per-Class AP — Test Set (green ≥ 0.85 | amber ≥ 0.70 | red < 0.70)")
ax.legend()
plt.xticks(rotation=45, ha="right", fontsize=9)

for bar, v in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.3f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/per_class_ap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Per-class AP chart → {PLOTS_DIR}/per_class_ap.png")

## Cell 22 — Per-class confusion matrices (binary, one per class)

In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(20, 12))
fig.suptitle("Per-Class Binary Confusion Matrices — Test Set", fontsize=14)
axes = axes.flatten()

for i, cls in enumerate(ALL_CLASSES):
    cm = confusion_matrix(test_labels[:, i], test_preds[:, i])
    sns.heatmap(
        cm, annot=True, fmt="d", ax=axes[i],
        cmap="Blues", cbar=False,
        xticklabels=["Pred 0", "Pred 1"],
        yticklabels=["True 0", "True 1"]
    )
    ap = metrics_tuned["per_class_ap"].get(cls, 0)
    axes[i].set_title(f"{cls}\nAP={ap:.3f}", fontsize=9)

# Hide unused subplots (13 classes, 15 subplots)
for j in range(len(ALL_CLASSES), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Confusion matrices → {PLOTS_DIR}/confusion_matrices.png")

## Cell 23 — Co-occurrence heatmap (predicted label pairs)

In [ ]:
cooc = test_preds.T @ test_preds      # (13, 13) — how often two classes predicted together
np.fill_diagonal(cooc, 0)             # remove self-co-occurrence

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(
    cooc, annot=True, fmt="d", ax=ax,
    xticklabels=ALL_CLASSES, yticklabels=ALL_CLASSES,
    cmap="YlOrRd", cbar=True
)
ax.set_title("Predicted Label Co-occurrence (test set)", fontsize=13)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/cooccurrence_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Co-occurrence heatmap → {PLOTS_DIR}/cooccurrence_heatmap.png")

## Cell 24 — Save full results JSON

In [ ]:
results = {
    "model"         : "ConvNeXt-Small",
    "task"          : "multi-label classification",
    "best_epoch"    : ckpt["epoch"],
    "val_mAP"       : ckpt["val_mAP"],
    "test_metrics_default_threshold"  : metrics_def,
    "test_metrics_tuned_threshold"    : metrics_tuned,
    "thresholds"    : thresh_dict,
    "classes"       : ALL_CLASSES,
    "img_size"      : IMG_SIZE,
    "mean"          : MEAN,
    "std"           : STD,
}

results_path = f"{OUTPUT_DIR}/test_results.json"
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)

print(f"✓ Results saved → {results_path}")
print(f"\nAll outputs:")
for p in Path(OUTPUT_DIR).rglob("*"):
    if p.is_file():
        size = p.stat().st_size / 1024
        print(f"  {str(p):<55} {size:>8.1f} KB")

## Cell 25 — Inference function (for deployment)

This is the function your FastAPI backend will call.  
Loads the checkpoint + thresholds and runs inference on a single PIL image.

In [ ]:
def load_inference_model(checkpoint_path: str,
                         threshold_path: str):
    """
    Load trained Model B for inference.
    Returns a predict() function that takes a PIL image.

    Usage:
        predict = load_inference_model(CHECKPOINT, THRESH_FILE)
        result  = predict(pil_image)
    """
    ckpt   = torch.load(checkpoint_path, map_location=DEVICE)
    model  = build_model()
    model.load_state_dict(ckpt["model_state"])
    model.to(DEVICE)
    model.eval()

    # Load per-class thresholds
    thresholds = np.full(NUM_CLASSES, DEFAULT_THRESH)
    if Path(threshold_path).exists():
        t_dict = json.load(open(threshold_path))
        thresholds = np.array([t_dict.get(cls, DEFAULT_THRESH)
                                for cls in ALL_CLASSES])

    transform = get_transforms("val")

    def predict(pil_image: Image.Image) -> dict:
        """
        Args:
            pil_image : PIL.Image — any size, will be resized

        Returns:
            {
                "detected"     : [{disease, confidence}, ...],  # above threshold
                "all_probs"    : {class: probability, ...},     # all 13 classes
                "multi_disease": bool,
                "is_healthy"   : bool,
                "severity"     : "Mild" | "Moderate" | "Severe" | "None"
            }
        """
        img = transform(pil_image.convert("RGB")).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs = torch.sigmoid(model(img)).cpu().numpy()[0]

        detected = []
        for cls, prob, thresh in zip(ALL_CLASSES, probs, thresholds):
            if prob >= thresh:
                detected.append({"disease": cls, "confidence": round(float(prob), 4)})
        detected.sort(key=lambda x: -x["confidence"])

        # Stage 1 severity estimate (replaced by Stage 2 in full pipeline)
        disease_only = [d for d in detected if d["disease"] != "Healthy"]
        if not disease_only:
            severity = "None"
        else:
            mc = max(d["confidence"] for d in disease_only)
            severity = "Mild" if mc < 0.65 else "Moderate" if mc < 0.85 else "Severe"

        return {
            "detected"     : detected,
            "all_probs"    : {cls: round(float(p), 4)
                              for cls, p in zip(ALL_CLASSES, probs)},
            "multi_disease": len(detected) > 1,
            "is_healthy"   : any(d["disease"] == "Healthy" for d in detected),
            "severity"     : severity,
        }

    return predict


# Test inference
print("Testing inference function...")
predict_fn = load_inference_model(CHECKPOINT, THRESH_FILE)

# Create a dummy image to test
dummy_img = Image.fromarray(
    (np.random.rand(224, 224, 3) * 255).astype(np.uint8)
)
result = predict_fn(dummy_img)

print(f"\nTest inference output:")
print(f"  Detected    : {result['detected']}")
print(f"  Multi-disease: {result['multi_disease']}")
print(f"  Is healthy  : {result['is_healthy']}")
print(f"  Severity    : {result['severity']}")
print(f"\n✓ Inference function ready for FastAPI deployment")

## Cell 26 — Summary

Final summary of everything produced by this notebook.

In [ ]:
print("=" * 65)
print("  MODEL B — MULTI-LABEL CLASSIFICATION — COMPLETE")
print("=" * 65)
print(f"\n  Architecture  : ConvNeXt-Small")
print(f"  Input size    : {IMG_SIZE}×{IMG_SIZE}")
print(f"  Classes       : {NUM_CLASSES}")
print(f"  Loss          : Asymmetric Loss (gamma_neg=4, gamma_pos=1)")
print(f"  Augmentation  : RandAugment + MixUp/CutMix (60% batches)")
print(f"  Training      : {WARMUP_EPOCHS} warmup + {EPOCHS-WARMUP_EPOCHS} finetune epochs")
print(f"  Best val mAP  : {best_mAP:.4f}")
print(f"\n  Test results (tuned thresholds):")
print(f"    mAP          : {metrics_tuned['mAP']:.4f}")
print(f"    Macro F1     : {metrics_tuned['macro_f1']:.4f}")
print(f"    Micro F1     : {metrics_tuned['micro_f1']:.4f}")
print(f"    Hamming loss : {metrics_tuned['hamming_loss']:.4f}")
print(f"    Exact match  : {metrics_tuned['exact_match']:.4f}")
print(f"\n  Output files:")
print(f"    {CHECKPOINT}")
print(f"    {THRESH_FILE}")
print(f"    {LOG_FILE}")
print(f"    {OUTPUT_DIR}/classification_report.txt")
print(f"    {OUTPUT_DIR}/test_results.json")
print(f"    {PLOTS_DIR}/training_curves.png")
print(f"    {PLOTS_DIR}/per_class_ap.png")
print(f"    {PLOTS_DIR}/confusion_matrices.png")
print(f"    {PLOTS_DIR}/cooccurrence_heatmap.png")
print(f"\n  Next step: plug predict_fn into FastAPI /predict endpoint")
print("=" * 65)